In [1]:
from vpython import *
from math import *
from random import *

scene = canvas()

N_A = 6.022e23	# [mol^-1] Avagadro's number
hbar = 1.05e-34	# [kg m^2 /s] Planck's reduced constant
kb = 1.38e-23	# [J/K] Boltzmann constant
l = 5e-10	# length of spring [m]  (separation of atoms)
dt = 1e-14	# [s] Time step
t = 0			# [s] Initial time
N = 3			# number of oscillators in each atom
atoms = []
totalS = 0.0
totalE = 0.0
totSLine=gcurve(color=color.yellow,label="total S",markers=True)

class atom:
    def __init__(self,m,k,rad,r,col,name="atom"):
        self.shape = sphere(color=col,radius=rad,pos=r)
        self.name = name
        self.m = m
        self.k = k
        self.dE = hbar*sqrt(self.k/self.m)
        self.ri = r #initial position
        self.energy = vec(self.dE/2,self.dE/2,self.dE/2)
        self.dq = [0,0,0]
        self.quanta = vec(0,0,0)
        self.quantaCount = 0
        self.f = vec(0,0,0)
        self.p = sqrt(self.m*(self.dE/2))*vec(1,1,1)
        self.line = gcurve(color=col,label=self.name,markers=True)
        atoms.append(self)
        self.label = label(pos=r, text=f'Quanta in {self.name}: '+str(self.quanta), border=1)
        self.omega = 0.0
        self.s = 0.0
        self.springs = []
        self.springs.append(helix(pos=r+vec(0,l,0), axis=vec(0,l,0), radius=0.5e-10))
        self.springs.append(helix(pos=r+vec(0,-l,0), axis=vec(0,-l,0), radius=0.5e-10))
        self.springs.append(helix(pos=r+vec(l,0,0), axis=vec(l,0,0), radius=0.5e-10))
        self.springs.append(helix(pos=r+vec(-l,0,0), axis=vec(-l,0,0), radius=0.5e-10))
        self.springs.append(helix(pos=r+vec(0,0,l), axis=vec(0,0,l), radius=0.5e-10))
        self.springs.append(helix(pos=r+vec(0,0,-l), axis=vec(0,0,-l), radius=0.5e-10))

    def click1(self,i):
        self.dq[i] = 1
        self.quanta += vec(self.dq[0],self.dq[1],self.dq[2])
        self.energy += vec(self.dq[0],self.dq[1],self.dq[2])*self.dE
        self.dq = [0,0,0]
        self.p = vec(sqrt(self.energy.x),sqrt(self.energy.y),sqrt(self.energy.z))*sqrt(2*self.m)
        self.label.text = f'Quanta in {self.name}: {str(self.quanta)}'
        self.quantaCount += 1
        self.omega = factorial(self.quantaCount+N-1)/(factorial(self.quantaCount)*factorial(N-1))
        
    def click2(self,i,q):#q = quantizable energy levels. Will be multiplied by 5 and rounded down to the next quantum level of this atom.
        self.dq[i] = int((5*q)/self.dE)
        self.quanta += vec(self.dq[0],self.dq[1],self.dq[2])
        self.energy += vec(self.dq[0],self.dq[1],self.dq[2])*self.dE
        self.p = vec(sqrt(self.energy.x),sqrt(self.energy.y),sqrt(self.energy.z))*sqrt(2*self.m)
        self.label.text = f'Quanta in {self.name}: {str(self.quanta)}'
        self.quantaCount += self.dq[i]
        self.dq = [0,0,0]
        self.omega = factorial(self.quantaCount+N-1)/(factorial(self.quantaCount)*factorial(N-1))#combin(N,self.quantaCount)
        
    def forceSpring(self):
        self.f = vec(0,0,0)
        for i in range(len(self.springs)):
            distance = self.shape.pos - self.springs[i].pos
            dL = mag(distance) - l
            self.f += -self.k*dL*norm(distance) #norm() returns unit vector
            self.springs[i].axis = distance
            
    def udPhys(self):
        global totalS, totalE
        self.forceSpring()
        self.p += self.f * dt
        self.v = self.p/self.m
        self.shape.pos += self.v*dt
        if self.omega>0:
            self.s = log(self.omega)*kb
        totalS += self.s
        totalE += mag(self.energy)
        
            

Al = atom(name="Al",m=26.98/(N_A*1000),k=2*16,rad=143e-12,r=vec(-l,0,0),col=color.blue)
Pb = atom(name="Pb",m=207.2/(N_A*1000),k=2*5,rad=181e-12,r=vec(+l,0,0),col=color.red)


def click1(evt):
    r = randint(0,2)
    for i in range(len(atoms)):
        atoms[i].click1(r)

def click2(evt):
    r = randint(0,2)
    for i in range(len(atoms)):
        atoms[i].click2(r,Al.dE)

b1 = button(pos=scene.caption_anchor, text='Single Quanta', bind=click1)
b2 = button(pos=scene.caption_anchor,text='Five Quanta', bind =click2)
g1 = graph(width=1000, title='Entropy vs. Energy', xtitle='Energy (J)', ytitle='Entropy (J/K)')  #think about what good values for xmin, xmax, ymin, ymax would be

while True:
    rate(60)
    totalS = 0.0
    totalE = 0.0
    for i in range(len(atoms)):
        atoms[i].udPhys()
    for i in range(len(atoms)):
        atoms[i].line.plot(mag(atoms[i].energy),atoms[i].s)
        totSLine.plot(totalE,totalS)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyboardInterrupt: 